In [3]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import triton
import triton.language as tl
from triton.testing import do_bench
import pandas as pd

# ---------------- config ----------------
SIZES        = [2**14, 2**16, 2**18, 2**20, 2**22, 2**24, 2**26]
RAW_ACTS     = ['gelu', 'silu', 'mish', 'gds_eager', 'gds']  # raw microbench
TRAIN_ACTS   = ['gelu', 'silu', 'mish', 'gds']               # training bench
GDS_INIT     = dict(a=0.3519, b=0.00219)   # or init_gelu/init_silu/init_mish values
EPOCHS       = 3
BATCH_SIZE   = 128
LR           = 0.05
AMP          = False
DATA_DIR     = './data'
NUM_WORKERS  = 4
# ----------------------------------------

assert torch.cuda.is_available(), "CUDA GPU required"
device = torch.device('cuda')
torch.backends.cudnn.benchmark = True
props = torch.cuda.get_device_properties(device)
print(f"GPU: {props.name} ({props.multi_processor_count} SMs) | torch {torch.__version__} | triton {triton.__version__}")

LOG2E = 1.4426950408889634  # 1/ln(2), folds exp -> hardware exp2

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE': 1024}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 2048}, num_warps=8),
        triton.Config({'BLOCK_SIZE': 4096}, num_warps=8),
    ], key=['n_elements'])
@triton.jit
def gaussian_decay_softplus_fwd_kernel(
    x_ptr, params_ptr, out_ptr, n_elements,
    ln2: tl.constexpr, log2e: tl.constexpr, MAX_EXP: tl.constexpr, BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    a = tl.load(params_ptr)
    b = tl.load(params_ptr + 1)
    softplus = tl.maximum(x, 0.0) + tl.log(1.0 + tl.exp2(-tl.abs(x) * log2e))
    exponent = tl.minimum(-a * x * x - b * x, MAX_EXP)
    g = tl.exp2(exponent * log2e)
    tl.store(out_ptr + offsets, softplus - ln2 * g, mask=mask)

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE': 1024}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_SIZE': 2048}, num_warps=8, num_stages=3),
        triton.Config({'BLOCK_SIZE': 4096}, num_warps=8, num_stages=2),
    ], key=['n_elements'])
@triton.jit
def gaussian_decay_softplus_bwd_kernel(
    x_ptr, params_ptr, grad_out_ptr, grad_x_ptr, grad_params_ptr, n_elements,
    ln2: tl.constexpr, log2e: tl.constexpr, MAX_EXP: tl.constexpr, BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(0)
    step = tl.num_programs(0) * BLOCK_SIZE
    a = tl.load(params_ptr)
    b = tl.load(params_ptr + 1)
    acc_a = 0.0
    acc_b = 0.0
    for start in range(pid * BLOCK_SIZE, n_elements, step):
        offsets = start + tl.arange(0, BLOCK_SIZE)
        mask = offsets < n_elements
        x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
        grad_out = tl.load(grad_out_ptr + offsets, mask=mask, other=0.0)
        exponent = tl.minimum(-a * x * x - b * x, MAX_EXP)
        t = ln2 * tl.exp2(exponent * log2e)
        sigmoid = tl.sigmoid(x)
        tl.store(grad_x_ptr + offsets,
                 grad_out * (sigmoid + t * (2.0 * a * x + b)), mask=mask)
        acc_a += tl.sum(grad_out * (t * x * x), axis=0)
        acc_b += tl.sum(grad_out * (t * x), axis=0)
    tl.atomic_add(grad_params_ptr, acc_a)
    tl.atomic_add(grad_params_ptr + 1, acc_b)

_SM_COUNT = {}
def _program_cap(device) -> int:
    idx = device.index or 0
    if idx not in _SM_COUNT:
        _SM_COUNT[idx] = torch.cuda.get_device_properties(device).multi_processor_count
    return 4 * _SM_COUNT[idx]

class GaussianDecaySoftplusTriton(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, params):
        if not x.is_contiguous():
            x = x.contiguous()
        ctx.save_for_backward(x, params)
        out = torch.empty_like(x)
        n_elements = x.numel()
        grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
        gaussian_decay_softplus_fwd_kernel[grid](
            x, params, out, n_elements, math.log(2.0), LOG2E, 80.0)
        return out

    @staticmethod
    def backward(ctx, grad_output):
        x, params = ctx.saved_tensors
        grad_output = grad_output.contiguous()
        n_elements = x.numel()
        grad_x = torch.empty_like(x)
        grad_params = torch.zeros_like(params)
        cap = _program_cap(x.device)
        grid = lambda meta: (min(triton.cdiv(n_elements, meta['BLOCK_SIZE']), cap),)
        gaussian_decay_softplus_bwd_kernel[grid](
            x, params, grad_output, grad_x, grad_params,
            n_elements, math.log(2.0), LOG2E, 80.0)
        return grad_x, grad_params

class Gaussian_pdf_softplus_v4(nn.Module):
    def __init__(self, a=0.3519, b=0.00219, params=None):
        super().__init__()
        # shared: pass one nn.Parameter to every instance;
        # standalone: each instance gets its own (a, b)
        self.params = params if params is not None else \
            nn.Parameter(torch.tensor([a, b], dtype=torch.float32))
    def forward(self, x):
        if x.dtype != torch.float32:                      # AMP: kernel stays fp32
            return GaussianDecaySoftplusTriton.apply(x.float(), self.params).to(x.dtype)
        return GaussianDecaySoftplusTriton.apply(x, self.params)
    def extra_repr(self):
        a, b = self.params.tolist()
        return f"a={a:.4f}, b={b:.4f}"

class GDSEager(nn.Module):
    def __init__(self, a=0.3519, b=0.00219, params=None):
        super().__init__()
        self.params = params if params is not None else \
            nn.Parameter(torch.tensor([a, b], dtype=torch.float32))
    def forward(self, x):
        a, b = self.params[0], self.params[1]
        return F.softplus(x) - math.log(2.0) * torch.exp(torch.clamp(-a*x*x - b*x, max=80.0))

def make_activation(name, shared_params=None):
    if name == 'gelu': return nn.GELU()
    if name == 'silu': return nn.SiLU()
    if name == 'mish': return nn.Mish()
    if name == 'gds': return Gaussian_pdf_softplus_v4(**GDS_INIT, params=shared_params)
    if name == 'gds_eager': return GDSEager(**GDS_INIT, params=shared_params)
    raise ValueError(name)

# ---- sanity check: Triton vs eager, fwd + grads ----
torch.manual_seed(0)
_mod = Gaussian_pdf_softplus_v4().to(device)
_x = torch.randn(1_000_000, device=device, requires_grad=True)
_g = torch.randn_like(_x)
_o_t, _o_e = _mod(_x), GDSEager().to(device)(_x)
_o_e = (F.softplus(_x) - math.log(2.0)*torch.exp(torch.clamp(
        -_mod.params[0]*_x*_x - _mod.params[1]*_x, max=80.0)))
gx_t, gp_t = torch.autograd.grad(_o_t, [_x, _mod.params], _g, retain_graph=True)
gx_e, gp_e = torch.autograd.grad(_o_e, [_x, _mod.params], _g)
print(f"fwd err {(_o_t-_o_e).abs().max():.2e} | dx err {(gx_t-gx_e).abs().max():.2e} "
      f"| dparams err {(gp_t-gp_e).abs().max():.2e}")
del _mod, _x, _g, _o_t, _o_e, gx_t, gp_t, gx_e, gp_e
torch.cuda.empty_cache()

GPU: NVIDIA GeForce RTX 5090 (170 SMs) | torch 2.8.0+cu128 | triton 3.4.0
fwd err 4.77e-07 | dx err 9.54e-07 | dparams err 4.30e+05


In [4]:
rows = []
for n in SIZES:
    x_base = torch.randn(n, device=device)
    g = torch.randn_like(x_base)
    elt = x_base.element_size()
    for name in RAW_ACTS:
        act = make_activation(name).to(device)

        with torch.no_grad():                                    # forward only
            fwd_ms = do_bench(lambda: act(x_base), warmup=25, rep=100)

        xx = x_base.clone().requires_grad_(True)                 # backward only
        out = act(xx)
        grads = [xx] + [p for p in act.parameters() if p.requires_grad]
        bwd_ms = do_bench(lambda: out.backward(g, retain_graph=True),
                          warmup=25, rep=100, grad_to_none=grads)

        fb_ms = do_bench(lambda: act(xx).backward(g),            # combined step
                         warmup=25, rep=100, grad_to_none=grads)

        rows.append(dict(elements=n, act=name,
                         fwd_ms=fwd_ms, bwd_ms=bwd_ms, fb_ms=fb_ms,
                         fwd_GBs=2*n*elt/(fwd_ms*1e-3)/1e9,      # read x, write out
                         bwd_GBs=3*n*elt/(bwd_ms*1e-3)/1e9))     # read x,g; write dx
        del act, xx, out
    del x_base, g
    torch.cuda.empty_cache()

raw_df = pd.DataFrame(rows)
ref = raw_df[raw_df.act == 'gelu'].set_index('elements').fb_ms
raw_df['fb_vs_gelu'] = raw_df.apply(lambda r: ref[r.elements] / r.fb_ms, axis=1)
raw_df.round(3)

,elements,act,fwd_ms,bwd_ms,fb_ms,fwd_GBs,bwd_GBs,fb_vs_gelu
0,16384,gelu,0.005,0.006,0.007,28.799,34.371,1.000
1,16384,silu,0.005,0.005,0.007,26.580,37.836,0.953
2,16384,mish,0.005,0.006,0.009,28.049,32.991,0.807
3,16384,gds_eager,0.024,0.559,0.923,5.368,0.352,0.007
4,16384,gds,0.005,0.007,0.271,28.617,26.499,0.025
5,65536,gelu,0.005,0.006,0.010,110.884,135.291,1.000
6,65536,silu,0.005,0.006,0.009,102.457,129.540,1.097
7,65536,mish,0.005,0.006,0.009,105.845,129.257,1.150
8,65536,gds_eager,0.025,0.121,0.467,21.233,6.489,0.021
9,65536,gds,0.005,0.019,0.269,106.792,42.014,0.037


In [9]:
from torchvision import datasets, transforms
from tqdm.auto import tqdm

# ---------------- training config (mirrors the lab's real runs) ----------------
EPOCHS          = 10        # lab uses 200; 20 is plenty for a speed comparison
BATCH_SIZE      = 512
LR              = 1e-3      # weights
WEIGHT_DECAY    = 0.1       # weights only
ACT_LR_MULT     = 0.05      # shape params lr = LR * this  (the NaN fix)
WARMUP_FRAC     = 0.05
GRAD_CLIP       = 1.0
LABEL_SMOOTHING = 0.1
# -------------------------------------------------------------------------------

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride, act):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(cout)
        self.act1, self.act2 = act(), act()
        self.short = nn.Sequential()
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(
                nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = self.act1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act2(out + self.short(x))

class ResNet18(nn.Module):
    """CIFAR variant: 3x3 stem, no maxpool, [2,2,2,2] blocks."""
    def __init__(self, act, num_classes=10):
        super().__init__()
        self.conv1, self.bn1, self.act = nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), act()
        self.layer1 = self._make(64, 64, 1, act)
        self.layer2 = self._make(64, 128, 2, act)
        self.layer3 = self._make(128, 256, 2, act)
        self.layer4 = self._make(256, 512, 2, act)
        self.fc = nn.Linear(512, num_classes)
        self.apply(self._init)
    @staticmethod
    def _make(cin, cout, stride, act):
        return nn.Sequential(BasicBlock(cin, cout, stride, act), BasicBlock(cout, cout, 1, act))
    @staticmethod
    def _init(m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        out = self.act(self.bn1(self.conv1(x)))
        out = self.layer4(self.layer3(self.layer2(self.layer1(out))))
        return self.fc(F.adaptive_avg_pool2d(out, 1).flatten(1))

def resnet18_cifar(act_name):
    shared = (nn.Parameter(torch.tensor([GDS_INIT['a'], GDS_INIT['b']], dtype=torch.float32))
              if act_name in ('gds', 'gds_eager') else None)
    return ResNet18(act=lambda: make_activation(act_name, shared))

def cosine_warmup_lr(base, epoch, epochs, warmup_frac=WARMUP_FRAC):
    warm = max(1, int(warmup_frac * epochs))
    if epoch < warm:
        return base * (epoch + 1) / warm
    t = (epoch - warm) / max(1, epochs - 1 - warm)
    return base * 0.5 * (1.0 + math.cos(math.pi * t))

norm = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
train_ld = torch.utils.data.DataLoader(
    datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
        transforms.ToTensor(), norm])),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
test_ld = torch.utils.data.DataLoader(
    datasets.CIFAR10(DATA_DIR, train=False, download=True,
                     transform=transforms.Compose([transforms.ToTensor(), norm])),
    batch_size=1024, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

def train_one(act_name):
    torch.manual_seed(0)
    model = resnet18_cifar(act_name).to(device)

    # lab-style param split: shape params get their own group, low lr, NO weight decay
    act_ids = {id(m.params) for m in model.modules()
               if isinstance(m, (Gaussian_pdf_softplus_v4, GDSEager))}
    act_params    = [p for p in model.parameters() if id(p) in act_ids]
    weight_params = [p for p in model.parameters() if id(p) not in act_ids]
    groups = [{'params': weight_params, 'lr': LR, 'weight_decay': WEIGHT_DECAY}]
    if act_params:
        groups.append({'params': act_params, 'lr': LR * ACT_LR_MULT, 'weight_decay': 0.0})
    opt = torch.optim.AdamW(groups)
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    torch.cuda.reset_peak_memory_stats(device)
    epoch_times = []
    for ep in range(EPOCHS):
        lr_w = cosine_warmup_lr(LR, ep, EPOCHS)
        opt.param_groups[0]['lr'] = lr_w
        if act_params:
            opt.param_groups[1]['lr'] = cosine_warmup_lr(LR * ACT_LR_MULT, ep, EPOCHS)

        model.train()
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for x, y in tqdm(train_ld, desc=f"{act_name} ep{ep+1}/{EPOCHS}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        torch.cuda.synchronize()
        epoch_times.append(time.perf_counter() - t0)
        if act_params:  # watch the shape params stay sane
            a, b = act_params[0].detach().cpu().tolist()
            print(f"    ep{ep+1}: loss {loss.item():.3f} | a={a:+.4f} b={b:+.4f} | {epoch_times[-1]:.1f}s")

    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in test_ld:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)

    steady = epoch_times[1:] or epoch_times
    res = dict(act=act_name,
               s_per_epoch=sum(steady)/len(steady),
               img_per_s=len(train_ld.dataset)/(sum(steady)/len(steady)),
               peak_mem_GB=torch.cuda.max_memory_allocated(device)/1e9,
               test_acc=100.0*correct/total,
               epoch_times=[round(t, 2) for t in epoch_times])
    del model, opt
    torch.cuda.empty_cache()
    return res

train_df = pd.DataFrame([train_one(a) for a in TRAIN_ACTS])
train_df['vs_gelu'] = train_df.img_per_s / train_df.loc[train_df.act == 'gelu', 'img_per_s'].iloc[0]
train_df.round(3)

gelu ep1/10:   0%|          | 0/97 [00:00<?, ?it/s]

gelu ep2/10:   0%|          | 0/97 [00:00<?, ?it/s]

gelu ep3/10:   0%|          | 0/97 [00:00<?, ?it/s]

gelu ep4/10:   0%|          | 0/97 [00:00<?, ?it/s]

gelu ep5/10:   0%|          | 0/97 [00:00<?, ?it/s]

...

gds ep9/10:   0%|          | 0/97 [00:00<?, ?it/s]

    ep9: loss 0.656 | a=+0.3711 b=-0.0162 | 4.2s


gds ep10/10:   0%|          | 0/97 [00:00<?, ?it/s]

    ep10: loss 0.691 | a=+0.3711 b=-0.0162 | 4.2s


,act,s_per_epoch,img_per_s,peak_mem_GB,test_acc,epoch_times,vs_gelu
0,gelu,4.193,11924.903,4.553,88.86,"[4.33, 4.2, 4.22, 4.18, 4.22, 4.18, 4.18, 4.18...",1.000
1,silu,4.191,11930.534,4.554,88.06,"[4.18, 4.18, 4.19, 4.2, 4.19, 4.18, 4.19, 4.19...",1.000
2,mish,4.224,11838.252,4.553,88.17,"[4.22, 4.28, 4.22, 4.22, 4.28, 4.18, 4.21, 4.2...",0.993
3,gds,4.229,11824.383,4.553,88.64,"[4.22, 4.22, 4.29, 4.23, 4.23, 4.23, 4.21, 4.2...",0.992


In [ ]:
_m = resnet18_cifar('gds')
_sites = [mod for mod in _m.modules() if isinstance(mod, Gaussian_pdf_softplus_v4)]
print(f"{len(_sites)} GDS sites, {len({id(m.params) for m in _sites})} unique param tensor")
print(f"optimizer sees {sum(p.numel() for p in _m.parameters())} params")  # only +2 total, not +18
del _m